# Phase 4 - Data Preparation

Project: **Sentiment Analysis with Facebook**

Clean the text, check duplicates, and create the leakage-free train / validation / test split.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
from src import config
from src.data.load_data import load_raw, inspect_dataset
from src.data.preprocess import prepare_dataframe, clean_text
from src.models.train import grouped_stratified_split

In [2]:
raw = load_raw()
df, prep = prepare_dataframe(raw)
prep

{'rows_initial': 5000,
 'dropped_missing_or_empty_text': 0,
 'dropped_invalid_label': 0,
 'dropped_exact_duplicate_records': 0,
 'dropped_empty_after_cleaning': 0,
 'rows_final': 5000,
 'unique_texts_final': 1345}

In [3]:
for t in raw['text'].head(5):
    print(repr(t), '->', repr(clean_text(t)))

'Mental health matters. Take a break. 🎉' -> 'mental health matters take break 🎉'
'Having trouble logging into my Stark account—any fix?' -> 'trouble logging stark account fix ?'
'When you open the fridge and forget why you came: 🔥 #relatable' -> 'open fridge forget came 🔥 relatable'
'In-depth: What market rally means for New York.' -> 'depth market rally means new york'
'Transfer rumor: J. Smith to City?' -> 'transfer rumor j smith city ?'


In [4]:
tr, va, te = grouped_stratified_split(df)
g = df['text_group'].values
print('overlap train/test groups:', len(set(g[tr]) & set(g[te])))
print('overlap train/val groups :', len(set(g[tr]) & set(g[va])))
for name, idx in [('train', tr), ('validation', va), ('test', te)]:
    print(name, len(idx), df.iloc[idx]['sentiment'].value_counts().to_dict())

overlap train/test groups: 0
overlap train/val groups : 0
train 3567 {'positive': 1608, 'neutral': 1363, 'negative': 596}
validation 717 {'positive': 323, 'neutral': 275, 'negative': 119}
test 716 {'positive': 322, 'neutral': 275, 'negative': 119}
